# Semantic Document Search with ChromaDB

Embedding an internal knowledge base with sentence-transformers, storing it in ChromaDB, and searching with metadata filters (department, active status).

_Datasets are not included in this repository._

In [24]:
!uv run pip install -q sentence-transformers chromadb

In [25]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

df = pd.read_csv('nusantarabank_kb.csv')
df.head()

,chunk_id,doc_title,department,is_active,text
0,CS-ATM-001,SOP Kartu Tertelan 2026,CS,True,"Jika kartu debit nasabah tertelan mesin ATM, b..."
1,CS-ATM-OLD,SOP Kartu Tertelan 2020,CS,False,"Jika kartu ATM tertelan, minta nasabah menungg..."
2,CS-CREDIT-001,Syarat Kartu Kredit,CS,True,Pengajuan kartu kredit Platinum mensyaratkan s...
3,HR-LEAVE-001,Kebijakan Cuti Melahirkan,HR,True,Karyawan wanita berhak atas cuti melahirkan se...
4,HR-LEAVE-OLD,Kebijakan Cuti Melahirkan 2015,HR,False,Cuti melahirkan diberikan selama 1.5 bulan seb...


In [26]:
df.tail()

,chunk_id,doc_title,department,is_active,text
2,CS-CREDIT-001,Syarat Kartu Kredit,CS,True,Pengajuan kartu kredit Platinum mensyaratkan s...
3,HR-LEAVE-001,Kebijakan Cuti Melahirkan,HR,True,Karyawan wanita berhak atas cuti melahirkan se...
4,HR-LEAVE-OLD,Kebijakan Cuti Melahirkan 2015,HR,False,Cuti melahirkan diberikan selama 1.5 bulan seb...
5,HR-RESIGN-001,Prosedur Resign,HR,True,Pemberitahuan pengunduran diri (One Month Noti...
6,IT-PWD-001,Reset Password Internal,IT,True,Reset kata sandi email kantor (@nusantarabank....


In [27]:
model_tr = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [28]:
ids = df["chunk_id"]

In [29]:
df["text"] = df["text"].astype(str)

In [30]:
df["text"] = list(df["text"])

In [31]:
document_embeddings = model_tr.encode(df["text"].tolist())

In [32]:
client = chromadb.Client()
collection = client.create_collection(name="bank_knowledge_base")

In [36]:
metadatas = df[["department", "is_active","doc_title"]].to_dict(orient="records")

In [37]:
collection.add(documents=df["text"].tolist(), embeddings=document_embeddings, ids=df["chunk_id"].tolist(), metadatas=metadatas)

In [49]:
def search_document(query_text, department):
    query_embedding = model_tr.encode([query_text])
    results = collection.query(query_embeddings=query_embedding, n_results=2, where={"$and":[{"department": department},{ "is_active": True}]})
    return results

In [38]:
query = "Kartu atm nasabah nyangkut di dalam mesin"

In [50]:
hasil = search_document(query_text=query, department="CS")

In [62]:
hasil["documents"][0]

['Jika kartu debit nasabah tertelan mesin ATM, blokir kartu segera melalui sistem, arahkan nasabah ke cabang terdekat dengan membawa KTP asli untuk pencetakan kartu baru bebas biaya.',
 'Pengajuan kartu kredit Platinum mensyaratkan slip gaji minimal Rp 15.000.000 per bulan dan histori BI Checking (SLIK) berstatus Kol 1.']

In [65]:
pd.DataFrame(hasil.get("documents")[0],columns=["text"])

,text
0,"Jika kartu debit nasabah tertelan mesin ATM, b..."
1,Pengajuan kartu kredit Platinum mensyaratkan s...
